# 03 — Modeling

We train three models of increasing flexibility — an interpretable **logistic
regression** baseline, a **random forest**, and **XGBoost** — and compare them on a
held-out validation set with **no leakage** (preprocessing is fit inside each
pipeline, on train only). We then run the imbalance ablation that justifies our
calibration choice.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config, data
pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (8, 4.5)

from src import model as M
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

df = data.load_raw()
X_train, X_val, X_test, y_train, y_val, y_test = data.split_data(df)

## Train and compare (validation set)

In [2]:
rows = []
fitted = {}
for kind in ['logreg','rf','xgboost']:
    m = M.build_model(kind).fit(X_train, y_train)
    fitted[kind] = m
    p = m.predict_proba(X_val)[:,1]
    rows.append({'model':kind,
                 'val_ROC_AUC':roc_auc_score(y_val,p),
                 'val_PR_AUC':average_precision_score(y_val,p),
                 'val_Brier':brier_score_loss(y_val,p)})
pd.DataFrame(rows).set_index('model').round(4)

,val_ROC_AUC,val_PR_AUC,val_Brier
model,,,
logreg,0.8456,0.3666,0.1490
rf,0.8554,0.3831,0.1390
xgboost,0.8589,0.3868,0.0494


**Read:** XGBoost has the best ranking (ROC-AUC / PR-AUC) *and* the
lowest Brier (best-calibrated). PR-AUC is the metric to watch under imbalance — the
no-skill baseline is just the positive rate (~0.067), so all three models add real
signal, with XGBoost ahead. We pick **XGBoost**.

## Imbalance ablation: does `scale_pos_weight` help?

The common reflex is to set `scale_pos_weight` (≈ neg/pos) to "fix" imbalance. Let's
measure what it actually does to **ranking** vs **calibration**.

In [3]:
spw = M.pos_weight(y_train)
xgb_plain = fitted['xgboost']
xgb_weighted = M.build_model('xgboost', scale_pos_weight=spw).fit(X_train, y_train)

for name, m in [('xgboost (plain)', xgb_plain), (f'xgboost (scale_pos_weight={spw:.1f})', xgb_weighted)]:
    p = m.predict_proba(X_val)[:,1]
    print(f'{name:42s}  ROC-AUC={roc_auc_score(y_val,p):.4f}  '
          f'PR-AUC={average_precision_score(y_val,p):.4f}  Brier={brier_score_loss(y_val,p):.4f}  '
          f'mean_pred={p.mean():.3f} (true={y_val.mean():.3f})')

xgboost (plain)                             ROC-AUC=0.8589  PR-AUC=0.3868  Brier=0.0494  mean_pred=0.067 (true=0.067)
xgboost (scale_pos_weight=14.0)             ROC-AUC=0.8578  PR-AUC=0.3906  Brier=0.1370  mean_pred=0.305 (true=0.067)


**Conclusion:** `scale_pos_weight` barely changes ranking (ROC/PR-AUC)
but **wrecks calibration** — mean predicted probability balloons far above the true
~6.7% default rate, and the Brier score worsens. Since we want a *calibrated risk
score* and we control the operating point with a cost-based threshold anyway, we keep
the **plain (calibrated) XGBoost** and move the threshold instead (notebook 04).

The full training pipeline (`python -m src.train`) refits this chosen model on the
training data and saves it to `models/model.joblib`.

**Next:** `04_evaluation` — honest test-set evaluation, the business threshold, and
SHAP explanations.